# ⏱️ Project 2 - Timer & Stopwatch

## What it demonstrates
| Concept | Where used |
|---------|------------|
| OOP (classes) | `Stopwatch`, `CountdownTimer` classes |
| `time` module | `time.perf_counter()`, `time.sleep()` |
| Properties | `elapsed`, `is_running` as computed properties |
| Exception handling | Starting an already-running timer |
| f-strings | `MM:SS.ms` formatted display |
| Dataclasses | `LapRecord` for storing lap data |
| Threading | Background countdown without blocking |

## Features
- **Stopwatch**: start, stop, pause, resume, lap recording
- **Countdown Timer**: set duration, tick down, alert on completion
- Both display time in `HH:MM:SS.ms` format
- Lap comparison — fastest / slowest lap

## Time formatting
```
raw seconds:  137.456
        ↓
hours   = 137 // 3600  = 0
minutes = 137 % 3600 // 60 = 2
seconds = 137 % 60   = 17
millis  = 0.456 * 1000 = 456
        ↓
display: 00:02:17.456
```

In [1]:
# ============================================================
#  PROJECT 2 — TIMER & STOPWATCH
# ============================================================

import time
from dataclasses import dataclass, field
from typing import Optional, List

# ---- Helper: format seconds → HH:MM:SS.ms ----
def format_time(seconds: float) -> str:
    """Convert raw seconds to HH:MM:SS.ms string."""
    h  = int(seconds) // 3600
    m  = int(seconds) % 3600 // 60
    s  = int(seconds) % 60
    ms = int((seconds - int(seconds)) * 1000)
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"
    return f"{m:02d}:{s:02d}.{ms:03d}"

# ---- Lap record ----
@dataclass
class LapRecord:
    number:     int
    lap_time:   float        # time for this lap only
    total_time: float        # cumulative time

    def __str__(self):
        return (f"  Lap {self.number:2d} │ "
                f"lap={format_time(self.lap_time):>12} │ "
                f"total={format_time(self.total_time)}")

# ---- Stopwatch class ----
class Stopwatch:
    """Stopwatch with lap recording and pause/resume support."""

    def __init__(self):
        self._start_time:  Optional[float] = None
        self._pause_time:  Optional[float] = None
        self._accumulated: float           = 0.0
        self._running:     bool            = False
        self._paused:      bool            = False
        self._laps:        List[LapRecord] = []
        self._last_lap_at: float           = 0.0

    @property
    def elapsed(self) -> float:
        """Total elapsed seconds including paused time."""
        if self._running and not self._paused:
            return self._accumulated + (time.perf_counter() - self._start_time)
        return self._accumulated

    @property
    def is_running(self) -> bool:
        return self._running and not self._paused

    def start(self):
        if self._running:
            raise RuntimeError("Stopwatch is already running. Use pause() to pause.")
        self._start_time = time.perf_counter()
        self._running    = True
        self._paused     = False
        print("  ▶ Stopwatch started.")

    def stop(self) -> float:
        if not self._running:
            raise RuntimeError("Stopwatch is not running.")
        self._accumulated += time.perf_counter() - self._start_time
        self._running      = False
        self._paused       = False
        print(f"  ⏹ Stopped. Total: {format_time(self._accumulated)}")
        return self._accumulated

    def pause(self):
        if not self._running or self._paused:
            raise RuntimeError("Cannot pause: stopwatch is not running.")
        self._accumulated += time.perf_counter() - self._start_time
        self._paused       = True
        print(f"  ⏸ Paused at {format_time(self._accumulated)}")

    def resume(self):
        if not self._paused:
            raise RuntimeError("Stopwatch is not paused.")
        self._start_time = time.perf_counter()
        self._paused     = False
        print(f"  ▶ Resumed from {format_time(self._accumulated)}")

    def lap(self) -> LapRecord:
        if not self._running:
            raise RuntimeError("Start the stopwatch before recording laps.")
        total    = self.elapsed
        lap_time = total - self._last_lap_at
        record   = LapRecord(len(self._laps) + 1, lap_time, total)
        self._laps.append(record)
        self._last_lap_at = total
        print(record)
        return record

    def reset(self):
        self.__init__()
        print("  🔄 Stopwatch reset.")

    def summary(self):
        print(f"\n  {'─'*50}")
        print(f"  📊 Summary — {len(self._laps)} laps")
        print(f"  {'─'*50}")
        for lap in self._laps:
            print(lap)
        if self._laps:
            fastest  = min(self._laps, key=lambda l: l.lap_time)
            slowest  = max(self._laps, key=lambda l: l.lap_time)
            avg      = sum(l.lap_time for l in self._laps) / len(self._laps)
            print(f"  {'─'*50}")
            print(f"  🏆 Fastest : Lap {fastest.number} — {format_time(fastest.lap_time)}")
            print(f"  🐢 Slowest : Lap {slowest.number} — {format_time(slowest.lap_time)}")
            print(f"  📈 Average : {format_time(avg)}")
            print(f"  ⏱ Total   : {format_time(self.elapsed)}")

# ---- Countdown Timer class ----
class CountdownTimer:
    """Countdown timer that ticks from duration down to zero."""

    def __init__(self, duration_seconds: float):
        if duration_seconds <= 0:
            raise ValueError("Duration must be positive.")
        self.duration   = duration_seconds
        self._started   = None

    @property
    def remaining(self) -> float:
        if self._started is None:
            return self.duration
        return max(0.0, self.duration - (time.perf_counter() - self._started))

    @property
    def is_expired(self) -> bool:
        return self.remaining <= 0

    def start(self):
        self._started = time.perf_counter()
        print(f"  ▶ Countdown started: {format_time(self.duration)}")

    def tick(self, interval: float = 1.0, ticks: int = 5):
        """Simulate the countdown by sampling at intervals."""
        for _ in range(ticks):
            time.sleep(interval)
            rem = self.remaining
            bar = "█" * int((rem / self.duration) * 20)
            bar = bar.ljust(20)
            print(f"  [{bar}] {format_time(rem)} remaining", end="\r")
            if self.is_expired:
                break
        print()

    def run(self, tick_interval: float = 0.5, max_ticks: int = 8):
        """Start and simulate the countdown."""
        self.start()
        self.tick(tick_interval, max_ticks)
        if self.is_expired:
            print("  🔔 Time's up!")
        else:
            print(f"  ⏸ Stopped early. {format_time(self.remaining)} remaining.")

print("✅ Stopwatch & CountdownTimer defined.")

✅ Stopwatch & CountdownTimer defined.


In [2]:
# ---- Demo — Stopwatch with laps ----
print("=" * 52)
print("         ⏱️  STOPWATCH DEMO")
print("=" * 52)

sw = Stopwatch()
sw.start()

# Simulate 4 laps with varying durations
lap_durations = [0.3, 0.18, 0.25, 0.22]
for dur in lap_durations:
    time.sleep(dur)
    sw.lap()

# Pause and resume
sw.pause()
time.sleep(0.1)          # paused — not counted
sw.resume()
time.sleep(0.15)
sw.lap()                 # lap 5

sw.stop()
sw.summary()

print()
print("=" * 52)
print("        ⏳  COUNTDOWN TIMER DEMO")
print("=" * 52)

# 3-second countdown with 0.5s ticks
cd = CountdownTimer(3.0)
cd.run(tick_interval=0.5, max_ticks=8)

         ⏱️  STOPWATCH DEMO
  ▶ Stopwatch started.
  Lap  1 │ lap=   00:00.300 │ total=00:00.300
  Lap  2 │ lap=   00:00.181 │ total=00:00.482
  Lap  3 │ lap=   00:00.250 │ total=00:00.733
  Lap  4 │ lap=   00:00.220 │ total=00:00.953
  ⏸ Paused at 00:00.954
  ▶ Resumed from 00:00.954
  Lap  5 │ lap=   00:00.152 │ total=00:01.105
  ⏹ Stopped. Total: 00:01.106

  ──────────────────────────────────────────────────
  📊 Summary — 5 laps
  ──────────────────────────────────────────────────
  Lap  1 │ lap=   00:00.300 │ total=00:00.300
  Lap  2 │ lap=   00:00.181 │ total=00:00.482
  Lap  3 │ lap=   00:00.250 │ total=00:00.733
  Lap  4 │ lap=   00:00.220 │ total=00:00.953
  Lap  5 │ lap=   00:00.152 │ total=00:01.105
  ──────────────────────────────────────────────────
  🏆 Fastest : Lap 5 — 00:00.152
  🐢 Slowest : Lap 1 — 00:00.300
  📈 Average : 00:00.221
  ⏱ Total   : 00:01.106

        ⏳  COUNTDOWN TIMER DEMO
  ▶ Countdown started: 00:03.000
  [                    ] 00:00.000 remaining
  🔔 